In [3]:
from qiskit import *
from math import log2
from typing import Optional
from qiskit.circuit.library import QFT

In [4]:
#cnot reduction

def get_divergence(a, b):
    n = int(log2(a)) + 1
    divergence = []
    for i in range(n):
        if ((a >> i) & 1) != ((b >> i) & 1):
            divergence.append(i)
    return divergence


def get_a1(a, b):
    divergence = get_divergence(a, b)
    div = divergence[1:]
    for idx in div:
        a = a ^ (1 << idx)
    return a, divergence


def get_a2(a, b):
    a1, divergence = get_a1(a, b)
    a2 = a1 ^ (1 << divergence[0])
    return a2


def pivot_divergence(piv_idx, divergence, n):
    quantum_circuit = QuantumCircuit(n)
    for idx in divergence:
        if idx != piv_idx:
            quantum_circuit.cx(piv_idx, idx)
    return quantum_circuit


def pivot_fix(piv_idx, divergence, a1, n):
    divergence.remove(piv_idx)
    controls = ""
    for idx in divergence:
        if((a1 >> idx) & 1):
            controls += "0"
        else:
            controls += "1"
    quantum_circuit = QuantumCircuit(n)
    quantum_circuit.mcx(divergence, piv_idx, ctrl_state=controls)
    return quantum_circuit


def cnot_reduction(a, b, n):

    if a == b: return QuantumCircuit(n)

    a1, divergence = get_a1(a, b)
    piv_idx = divergence[0]

    quantum_circuit = QuantumCircuit(n)
    quantum_circuit.name = f"{a}⟷{b}"
    
    quantum_circuit.append(pivot_divergence(piv_idx, divergence, n), range(n))
    quantum_circuit.append(pivot_fix(piv_idx, divergence, a1, n), range(n))
    quantum_circuit.append(pivot_divergence(piv_idx, divergence, n), range(n))

    return quantum_circuit

In [5]:
#point arithmetic

Point = tuple[int, int]

def mod_inverse(n: int, p: int) -> int:
    """Modular inverse via Fermat's little theorem (p must be prime)."""
    return pow(n, p - 2, p)


def point_add(P: Point, Q: Point, a: int, p: int) -> Point:
    """
    Add two points P and Q on the elliptic curve y² = x³ + ax + b (mod p).
    Returns the resulting point, or (0, 0) if the result is the point at infinity.
    """
    # Identidade: se um dos pontos for o infinito (0, 0), retorna o outro
    if P == (0, 0):
        return Q
    if Q == (0, 0):
        return P

    x1, y1 = P
    x2, y2 = Q

    # P + (-P) = infinity
    if x1 == x2 and (y1 + y2) % p == 0:
        return (0, 0)

    if P == Q:
        # Point doubling
        if y1 == 0:
            return (0, 0)
        lam = (3 * x1 * x1 + a) * mod_inverse(2 * y1, p) % p
    else:
        # Point addition
        lam = (y2 - y1) * mod_inverse(x2 - x1, p) % p

    x3 = (lam * lam - x1 - x2) % p
    y3 = (lam * (x1 - x3) - y1) % p
    return (x3, y3)


def scalar_mult(k: int, P: Point, a: int, p: int) -> Point:
    """Compute k·P using the double-and-add algorithm."""
    # O elemento neutro (identidade) agora é (0, 0)
    result: Point = (0, 0)  
    addend = P
    
    while k:
        if k & 1:
            result = point_add(result, addend, a, p)
        addend = point_add(addend, addend, a, p)
        k >>= 1
        
    return result

def is_on_curve(P: Point, curve_params: tuple[int, int, int]) -> bool:
    """
    Verifica se um ponto P(x, y) pertence à curva elíptica y² = x³ + ax + b (mod p).
    Recebe o Ponto e uma tupla contendo os parâmetros (a, b, p).
    """
    a, b, p = curve_params
    
    if P == (0, 0):
        return True
        
    x, y = P
    
    lhs = (y * y) % p
    rhs = (x * x * x + a * x + b) % p
    
    return lhs == rhs

def curve_points(a: int, b: int, p: int) -> list[Point]:
    """Enumerate all affine points on the curve, including the point at infinity in (0, 0)."""
    points: list[Point] = [(0, 0)]
    for x in range(p):
        rhs = (x * x * x + a * x + b) % p
        for y in range(p):
            if (y * y) % p == rhs:
                points.append((x, y))
    return points

In [ ]:
#mod inv transposition

def mod_inverse(n: int, p: int) -> int:
    """Modular inverse via Fermat's little theorem (p must be prime)."""
    return pow(n, p - 2, p)

def get_inverse_transpositions(p: int):
    transpositions = {}

    for i in range(1, p):
        if transpositions.get(i) is not None:
            continue
        inv = mod_inverse(i, p)
        transpositions[i] = inv
        transpositions[inv] = i
    
    return transpositions

def build_iverse_transposition_circuit(p: int, n: int):
    transpositions = get_inverse_transpositions(p)
    circuits = []
    for i, j in transpositions.items():
        circuit = cnot_reduction(i, j, n)
        circuits.append(circuit)
    return circuits

def complete_inverse_transposition(p: int, n: int):
    reg_a = QuantumRegister(n, name="a")
    quantum_circuit = QuantumCircuit(reg_a, name="complete_inverse_transposition")

    circuits = build_iverse_transposition_circuit(p, n)
    for circuit in circuits:
        quantum_circuit.append(circuit, reg_a)

    return quantum_circuit

In [7]:
#adders

from math import ceil, floor

def carry_with_const(c: int, n: int):
    """
    computes the carry of the sum of a constant c to an n-bit quantum register.
    result outbut is a single qubit.
    for the circuit construction, we use n-1 dirty ancilla qubits in an arbitrary state |g>, that are returned to the same state at the end of the computation.
    """
    reg_a = QuantumRegister(n, name='a')
    reg_g = QuantumRegister(n-1, name='g')
    reg_carry = QuantumRegister(1, name='carry')
    quantum_circuit = QuantumCircuit(reg_a, reg_g, reg_carry, name="carry_with_const")

    if n == 1:
        if c & 1:
            quantum_circuit.cx(reg_a[0], reg_carry[0])
        return quantum_circuit.to_gate()

    quantum_circuit.cx(reg_g[n-2], reg_carry[0])

    for i in range(n - 2, -1, -1):
        if (c >> (i+1)) & 1:
            quantum_circuit.cx(reg_a[i+1], reg_g[i])
            quantum_circuit.x(reg_a[i+1])
        if i == 0: continue
        quantum_circuit.ccx(reg_g[i-1], reg_a[i+1], reg_g[i])

    if(c&1):
        quantum_circuit.ccx(reg_a[0], reg_a[1], reg_g[0])

    for i in range(n - 2):
        quantum_circuit.ccx(reg_g[i], reg_a[i+2], reg_g[i+1])

    quantum_circuit.cx(reg_g[n-2], reg_carry[0])

    for i in range(n - 3, -1, -1):
        quantum_circuit.ccx(reg_g[i], reg_a[i+2], reg_g[i+1])

    if(c&1):
        quantum_circuit.ccx(reg_a[0], reg_a[1], reg_g[0])

    for i in range(n - 1):
            if i != 0:
                quantum_circuit.ccx(reg_g[i-1], reg_a[i+1], reg_g[i])
            if (c >> (i+1)) & 1:
                quantum_circuit.x(reg_a[i+1])
                quantum_circuit.cx(reg_a[i+1], reg_g[i])
                
    return quantum_circuit.to_gate()

def sub_carry_gate():
    quantum_circuit1 = QuantumCircuit(3)
    quantum_circuit1.cx(0,1)
    quantum_circuit1.cx(2,0)
    quantum_circuit1.ccx(0,1,2)

    quantum_circuit2 = QuantumCircuit(3)
    quantum_circuit2.ccx(0,1,2)
    quantum_circuit2.cx(2,0)
    quantum_circuit2.cx(2,1)

    return (quantum_circuit1.to_gate(), quantum_circuit2.to_gate())

def incrementer(n:int):
    reg_v = QuantumRegister(n, name="v")
    reg_g = QuantumRegister(n, name="g")
    quantum_circuit = QuantumCircuit(reg_g, reg_v, name="incrementer")

    for i in range(n): 
        quantum_circuit.cx(reg_g[0], reg_v[i])
        if i != 0: quantum_circuit.x(reg_g[i])
    quantum_circuit.x(reg_v[n-1])

    sub_gate1, sub_gate2 = sub_carry_gate()

    for i in range(n-1):
        quantum_circuit.append(sub_gate1, reg_g[i:i+1] + reg_v[i:i+1] + reg_g[i+1:i+2])

    quantum_circuit.cx(reg_g[n-1], reg_v[n-1])

    for i in range(n-2, -1, -1):
        quantum_circuit.append(sub_gate2, reg_g[i:i+1] + reg_v[i:i+1] + reg_g[i+1:i+2])

    for i in range(1, n): quantum_circuit.x(reg_g[i])

    for i in range(n-1):
        quantum_circuit.append(sub_gate1, reg_g[i:i+1] + reg_v[i:i+1] + reg_g[i+1:i+2])

    quantum_circuit.cx(reg_g[n-1], reg_v[n-1])

    for i in range(n-2, -1, -1):
        quantum_circuit.append(sub_gate2, reg_g[i:i+1] + reg_v[i:i+1] + reg_g[i+1:i+2])

    for i in range(n): 
        quantum_circuit.cx(reg_g[0], reg_v[i])

    return quantum_circuit.to_gate()
    

def recursive_construction(n: int, c: int):

    reg_x = QuantumRegister(n, name='x')
    reg_anc = QuantumRegister(1, name='anc')
    quantum_circuit = QuantumCircuit(reg_x, reg_anc)

    low_bits = ceil(n/2)
    high_bits = floor(n/2)
    low_class = c & ((1 << low_bits) - 1)
    high_class = c >> low_bits

    if n == 1:
        quantum_circuit.x(reg_x[0]) if c & 1 else None
        return quantum_circuit.to_gate()

    quantum_circuit.append(carry_with_const(low_class, low_bits), reg_x[:low_bits] + reg_x[low_bits:2*low_bits-1] + reg_anc[:])
    quantum_circuit.append(incrementer(high_bits).control(1), reg_anc[:] + reg_x[:high_bits] + reg_x[low_bits:])
    quantum_circuit.append(carry_with_const(low_class, low_bits).inverse(), reg_x[:low_bits] + reg_x[low_bits:2*low_bits-1] + reg_anc[:])

    quantum_circuit.append(recursive_construction(low_bits, low_class), reg_x[:low_bits] + reg_anc[:])
    quantum_circuit.append(recursive_construction(high_bits, high_class), reg_x[low_bits:] + reg_anc[:])

    return quantum_circuit.to_gate()

def haner_class_adder(c: int, n: int):
    reg_x = QuantumRegister(n, name='x')
    reg_anc = QuantumRegister(1, name='anc')

    quantum_circuit = QuantumCircuit(reg_x, reg_anc, name="class_adder")
    quantum_circuit = recursive_construction(n, c)

    return quantum_circuit

def qc_MAJ() -> QuantumCircuit:
    quantum_circuit = QuantumCircuit(3)
    quantum_circuit.name = "MAJ"
    quantum_circuit.cx(2, 1)
    quantum_circuit.cx(2, 0)
    quantum_circuit.ccx(0, 1, 2)
    return quantum_circuit

def qc_UMA(two_version: bool=True) -> QuantumCircuit:
    #if argument is true, implements the 2cnot version, else, implements the 3cnot version
    #check reference
    quantum_circuit = QuantumCircuit(3)
    quantum_circuit.name = "UMA"
    if two_version:
        quantum_circuit.ccx(0, 1, 2)
        quantum_circuit.cx(2, 0)
        quantum_circuit.cx(0, 1)
    else:
        quantum_circuit.x(1)
        quantum_circuit.cx(0, 1)
        quantum_circuit.ccx(0, 1, 2)
        quantum_circuit.x(1)
        quantum_circuit.cx(2, 0)
        quantum_circuit.cx(2, 1)
    return quantum_circuit

def complement(n: int) -> QuantumCircuit:
    reg_a = QuantumRegister(n, name="a")
    quantum_circuit = QuantumCircuit(reg_a, name="complement")
    quantum_circuit.x(reg_a)
    return quantum_circuit

def comparator_CDKM(n: int) -> QuantumCircuit:
    reg_anc = QuantumRegister(1, name="anc")
    reg_a = QuantumRegister(n, name="a")
    reg_b = QuantumRegister(n, name="b")
    reg_s = QuantumRegister(1, name="s")
    quantum_circuit = QuantumCircuit(reg_anc, reg_a, reg_b, reg_s, name="comparator_CDKM")

    quantum_circuit.append(complement(n), reg_a[:])

    quantum_circuit.append(qc_MAJ(), [reg_anc[0], reg_b[0], reg_a[0]])

    for i in range(1, n):
        quantum_circuit.append(qc_MAJ(), [reg_a[i-1], reg_b[i], reg_a[i]])

    quantum_circuit.cx(reg_a[n-1], reg_s[0])

    for i in range(n-1, 0, -1):
            quantum_circuit.append(qc_MAJ().inverse(), [reg_a[i-1], reg_b[i], reg_a[i]])

    quantum_circuit.append(qc_MAJ().inverse(), [reg_anc[0], reg_b[0], reg_a[0]])
    
    quantum_circuit.append(complement(n), reg_a[:])
    #quantum_circuit.x(reg_s[0])

    return quantum_circuit

def adder_CDKM(num_qubits: int, modulo_2n: bool=False) -> QuantumCircuit:
    c = QuantumRegister(1, name="c")
    a = QuantumRegister(num_qubits, name="a")
    b = QuantumRegister(num_qubits, name="b")
    if not modulo_2n:
        z = QuantumRegister(1, name="z") 
        quantum_circuit = QuantumCircuit(c,a,b,z, name="Adder-CDKM")
    else:
        quantum_circuit = QuantumCircuit(c,a,b, name="Adder-CDKM-MOD2^n")
    
    quantum_circuit.append(qc_MAJ(), c[0:1] + b[0:1] + a[0:1])

    for i in range(1, num_qubits):
        quantum_circuit.append(qc_MAJ(), a[i-1:i] + b[i:i+1] + a[i:i+1])

    if not modulo_2n: quantum_circuit.cx(a[-1], z[0])
        
    for i in range(num_qubits-1, 0, -1):
        quantum_circuit.append(qc_UMA(), a[i-1:i] + b[i:i+1] + a[i:i+1])
        
    quantum_circuit.append(qc_UMA(), c[0:1] + b[0:1] + a[0:1])
    return quantum_circuit

def adder_mod(n: int, p: int) -> QuantumCircuit:
    reg_x = QuantumRegister(n, name="x")
    reg_y = QuantumRegister(n, name="y")
    reg_anc = QuantumRegister(2, name="anc")

    quantum_circuit = QuantumCircuit(reg_x, reg_y, reg_anc)

    quantum_circuit.append(adder_CDKM(n), reg_anc[1:2] + reg_x[:] + reg_y[:] + reg_anc[0:1])
    quantum_circuit.append(haner_class_adder(p, n+1).inverse(), reg_y[:] + reg_anc[:])
    quantum_circuit.append(haner_class_adder(p, n).control(1), reg_anc[0:1] + reg_y[:] + reg_anc[1:2])
    quantum_circuit.append(comparator_CDKM(n), reg_anc[1:2] + reg_y[:] + reg_x[:] + reg_anc[0:1])
    quantum_circuit.x(reg_anc[0])
    
    return quantum_circuit

In [8]:
#mult 

from math import log2

def mult_by_2(n:int):
    """
    performs multiplication by 2 using a cyclic bit shifts.
    n is the number of bits of the register.
    """

    qc = QuantumCircuit(n+1, name="mult_by_2")

    for i in range(n-1, -1, -1):
        qc.swap(i, i+1)
    return qc

def double_mod(N:int):
    n = int(log2(N)) + 1
    reg_x = QuantumRegister(n, "x")
    reg_anc = QuantumRegister(2, "anc")
    quantum_circuit = QuantumCircuit(reg_x, reg_anc, name="double_mod")
    
    quantum_circuit.append(mult_by_2(n), reg_x[:] + reg_anc[0:1])
    quantum_circuit.append(haner_class_adder(N, n+1).inverse(), reg_x[:] + reg_anc[:])
    quantum_circuit.append(haner_class_adder(N, n).control(1), reg_anc[0:1] + reg_x[:] + reg_anc[1:2])
    quantum_circuit.cx(reg_x[0], reg_anc[0], ctrl_state="0")

    return quantum_circuit

def qq_mult_mod(n: int, p: int) -> QuantumCircuit:
    reg_x = QuantumRegister(n, name="x")
    reg_y = QuantumRegister(n, name="y")
    reg_s = QuantumRegister(n, name="s")
    reg_anc = QuantumRegister(2, name="anc")

    quantum_circuit = QuantumCircuit(reg_x, reg_y, reg_s, reg_anc, name="qq_mult_mod")

    for i in range(n-1, -1, -1):
        quantum_circuit.append(adder_mod(n, p).control(1), reg_x[i:i+1] + reg_y[:] + reg_s[:] + reg_anc[:])
        if i != 0: quantum_circuit.append(double_mod(p), reg_s[:] + reg_anc[:])

    return quantum_circuit

In [9]:
#square

def square_mod(n: int, p: int) -> QuantumCircuit:
    reg_x = QuantumRegister(n, name="x")
    reg_s = QuantumRegister(n, name="s")
    reg_anc = QuantumRegister(3, name="anc")
    quantum_circuit = QuantumCircuit(reg_x, reg_s, reg_anc, name="square_mod")

    for i in range(n-1, -1, -1):
        quantum_circuit.cx(reg_x[i], reg_anc[2])
        quantum_circuit.append(adder_mod(n, p).control(1), reg_anc[2:3] + reg_x[:] + reg_s[:] + reg_anc[0:2])
        if i != 0: quantum_circuit.append(double_mod(p), reg_s[:] + reg_anc[0:2])
        quantum_circuit.cx(reg_x[i], reg_anc[2])

    return quantum_circuit

In [12]:
def cnot_all(n: int):
    reg_a = QuantumRegister(n, name="a")
    reg_b = QuantumRegister(n, name="b")

    quantum_circuit = QuantumCircuit(reg_a, reg_b, name="cnot_all")
    for i in range(n):
        quantum_circuit.cx(reg_a[i], reg_b[i])

    return quantum_circuit

def placeholder(n: int):
    reg_a = QuantumRegister(n, name="a")
    quantum_circuit = QuantumCircuit(reg_a, name="placeholder")
    return quantum_circuit

In [ ]:
def mod_negation(n: int, p: int):
    """
    Computes the negative modulo p of a quabntum register |x⟩ -> |-x mod p⟩.
    To get this result we make use of the equality -x mod p = p - x, for x <= p.
    Moreover, we use the identity p-x = (p' + x)', where a' is the bitwise complement, transforming the subtraction into a sum that can be done by using 
    the classical-quantum adder. 
    """
    reg_x = QuantumRegister(n, name="x")
    reg_anc = QuantumRegister(1, name="anc")

    quantum_circuit = QuantumCircuit(reg_x, reg_anc, name="mod_neg")

    p_compl = p ^ (2**(int(log2(p)) + 1) - 1)

    quantum_circuit.append(haner_class_adder(p_compl, n), reg_x[:] + reg_anc[:])
    quantum_circuit.append(complement(n), reg_x[:])

    return quantum_circuit


In [ ]:
def complement(n: int) -> QuantumCircuit:
    reg_a = QuantumRegister(n, name="a")
    quantum_circuit = QuantumCircuit(reg_a, name="complement")
    quantum_circuit.x(reg_a)
    return quantum_circuit

def comparator_class(c: int, n: int):
    """
    compares the value of a quantum_register x with a classical value c, 
    it toggles the carry bit if x < c.
    """

    reg_a = QuantumRegister(n, name='a')
    reg_g = QuantumRegister(n-1, name='g')
    reg_carry = QuantumRegister(1, name='carry')
    quantum_circuit = QuantumCircuit(reg_a, reg_g, reg_carry)

    quantum_circuit.append(complement(n), reg_a[:])

    if n == 1:
        if c & 1:
            quantum_circuit.cx(reg_a[0], reg_carry[0])
        return quantum_circuit

    quantum_circuit.cx(reg_g[n-2], reg_carry[0])

    for i in range(n - 2, -1, -1):
        if (c >> (i+1)) & 1:
            quantum_circuit.cx(reg_a[i+1], reg_g[i])
            quantum_circuit.x(reg_a[i+1])
        if i == 0: continue
        quantum_circuit.ccx(reg_g[i-1], reg_a[i+1], reg_g[i])

    if(c&1):
        quantum_circuit.ccx(reg_a[0], reg_a[1], reg_g[0])

    for i in range(n - 2):
        quantum_circuit.ccx(reg_g[i], reg_a[i+2], reg_g[i+1])

    quantum_circuit.cx(reg_g[n-2], reg_carry[0])

    for i in range(n - 3, -1, -1):
        quantum_circuit.ccx(reg_g[i], reg_a[i+2], reg_g[i+1])

    if(c&1):
        quantum_circuit.ccx(reg_a[0], reg_a[1], reg_g[0])

    for i in range(n - 1):
            if i != 0:
                quantum_circuit.ccx(reg_g[i-1], reg_a[i+1], reg_g[i])
            if (c >> (i+1)) & 1:
                quantum_circuit.x(reg_a[i+1])
                quantum_circuit.cx(reg_a[i+1], reg_g[i])

    quantum_circuit.append(complement(n), reg_a[:])
                
    return quantum_circuit

from math import log2

def haner_mod_class_adder(c: int, n: int, p: int):
    reg_x = QuantumRegister(n, name='x')
    reg_g = QuantumRegister(n-1, name='g')
    reg_anc = QuantumRegister(1, name='anc')

    quantum_circuit = QuantumCircuit(reg_x, reg_g, reg_anc, name="mod_class_adder")

    quantum_circuit.append(comparator_class(p-c, n), reg_x[:] + reg_g[:] + reg_anc[:])

    quantum_circuit.append(haner_class_adder(c, n, dirty=True).control(1), reg_anc[:] + reg_x[:] + reg_g[0:1]) 
    quantum_circuit.append(haner_class_adder(p-c, n, dirty=True).inverse().control(1, ctrl_state="0"), reg_anc[:] + reg_x[:] + reg_g[0:1])

    quantum_circuit.append(comparator_class(c, n), reg_x[:] + reg_g[:] + reg_anc[:])
    quantum_circuit.x(reg_anc)

    return quantum_circuit

In [ ]:
def elc_sum_1(Q: Point, params: tuple[int, int, int]):
    """
    implement case 1 of elliptic curve point addition in quantum circuits where one of the points is classically known.
    case 1 specifies that the two points are distinct and not inverses of each other, and neither is the point at infinity.
    """
    assert is_on_curve(Q, params), "Point Q is not on the curve."

    a, b, p = params
    x2, y2 = Q

    n = int(log2(p)) + 1

    reg_ctrl = QuantumRegister(1, name="ctrl")
    reg_x = QuantumRegister(n, name="x1")
    reg_y = QuantumRegister(n, name="y1")
    reg_t = QuantumRegister(n, name="t")
    reg_lam = QuantumRegister(n, name="λ")
    reg_anc = QuantumRegister(2, name="anc")

    quantum_circuit = QuantumCircuit(reg_ctrl, reg_x, reg_y, reg_t, reg_lam, reg_anc, name="elc_sum_1")

    #sub class coordinates, using regs t and lam as ancillas
    quantum_circuit.append(haner_mod_class_adder(x2, n, p).inverse(), reg_x[:] + reg_t[:n-1] + reg_anc[0:1]) 
    quantum_circuit.append(haner_mod_class_adder(y2, n, p).inverse().control(1), reg_ctrl[:] + reg_y[:] + reg_lam[:n-1] + reg_anc[1:2])

    #get inverse of x1 - x2
    quantum_circuit.append(cnot_all(n), reg_x[:] + reg_t[:])
    quantum_circuit.append(complete_inverse_transposition(p, n), reg_t[:])

    #multiplication to get λ
    quantum_circuit.append(qq_mult_mod(n, p), reg_y[:] + reg_t[:] + reg_lam[:] + reg_anc[:])

    #reset register y1
    quantum_circuit.append(qq_mult_mod(n, p).inverse(), reg_lam[:] + reg_x[:] + reg_y[:] + reg_anc[:])

    #reset register t
    quantum_circuit.append(complete_inverse_transposition(p, n).inverse(), reg_t[:])
    quantum_circuit.append(cnot_all(n), reg_x[:] + reg_t[:])

    #t gets λ^2
    quantum_circuit.append(square_mod(n, p), reg_lam[:] + reg_t[:] + reg_anc[:] + reg_y[0:1]) #using the freshly reset y1 qubit as ancilla for the square_mod circuit

    #sub λ^2 from x1
    quantum_circuit.append(adder_mod(n, p).inverse(), reg_t[:] + reg_x[:] + reg_anc[:])

    #controlled sum of 3x2 in x1, using reg t as ancilla
    quantum_circuit.append(haner_mod_class_adder((3 * x2) % p, n, p).control(1), reg_ctrl[:] + reg_x[:] + reg_t[:n-1] + reg_anc[0:1])

    #resetting t
    quantum_circuit.append(square_mod(n, p).inverse(), reg_lam[:] + reg_t[:] + reg_anc[:] + reg_y[0:1])

    #mult λ and x1 into y1
    quantum_circuit.append(qq_mult_mod(n, p), reg_lam[:] + reg_x[:] + reg_y[:] + reg_anc[:])

    #inverse of x1 into t
    quantum_circuit.append(cnot_all(n), reg_x[:] + reg_t[:])
    quantum_circuit.append(complete_inverse_transposition(p, n), reg_t[:])

    #mult t and y1 into λ, λ reset
    quantum_circuit.append(qq_mult_mod(n, p), reg_t[:] + reg_y[:] + reg_lam[:] + reg_anc[:])

    #reset t
    quantum_circuit.append(complete_inverse_transposition(p, n).inverse(), reg_t[:])
    quantum_circuit.append(cnot_all(n), reg_x[:] + reg_t[:])

    #modular negation of x1 register
    quantum_circuit.append(mod_negation(n, p), reg_x[:] + reg_anc[0:1]) #placeholder for modular negation of x1 register

    #controlled subtraction of y2 from y1 result is y3, using reg lam as ancilla
    quantum_circuit.append(haner_mod_class_adder(y2, n, p).inverse().control(1), reg_ctrl[:] + reg_y[:] + reg_lam[:n-1] + reg_anc[1:2])

    #adding x2 to x1, result is x3, using reg t as ancilla
    quantum_circuit.append(haner_mod_class_adder(x2, n, p), reg_x[:] + reg_t[:n-1] + reg_anc[0:1])

    return quantum_circuit